# PHIA Single-Agent Baseline

This notebook demonstrates the **PHIA (Personal Health Insights Agent)** baseline - the original single-agent architecture that PHA improves upon.

## Research Question
**"Is specialization better than one generalist?"**

## Architecture Comparison

| Aspect | PHIA (This Baseline) | PHA (Multi-Agent) |
|--------|---------------------|---------------------|
| Agents | 1 generalist | 3 specialists + orchestrator |
| Routing | None | Intelligent |
| Prompts | 1 combined | 4 specialized |
| Tools | Shared pool | Agent-specific |

## How PHIA Works

```
User Query
     ↓
┌─────────────────────────────────────┐
│      PHIA Single ReAct Agent        │
│                                     │
│  Tools:                             │
│  - tool_code (Python/pandas)        │
│  - search (web search)              │
│  - finish (final answer)            │
│                                     │
│  Prompt: Combined preamble with     │
│  DataFrame schemas + health context │
└──────────────┬──────────────────────┘
               ↓
         Final Response
```

## Setup

In [ ]:
import os
import sys

# Set API keys (replace with your own or set environment variables)
# os.environ['GEMINI_API_KEY'] = 'your-key-here'
# os.environ['TAVILY_API_KEY'] = 'your-key-here'  # Optional, for web search

# Add parent directory to path if running from notebooks folder
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

# Check onetwo availability
from pha.agents import is_onetwo_available

if not is_onetwo_available():
    print("⚠️ onetwo is not available. PHIA baseline requires onetwo.")
    print("Install with: pip install onetwo")
else:
    print("✓ onetwo is available")

In [ ]:
# API Keys
GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY') or input('Enter Gemini API key: ')
TAVILY_API_KEY = os.environ.get('TAVILY_API_KEY') or input('Enter Tavily API key (optional): ') or None

print(f"Gemini API key: {'✓ Set' if GEMINI_API_KEY else '✗ Missing'}")
print(f"Tavily API key: {'✓ Set' if TAVILY_API_KEY else '✗ Missing (web search disabled)'}")

In [ ]:
# Load settings and data
from config.settings import Settings

settings = Settings()
print(f"Data directory: {settings.data_dir}")
print(f"Summary path: {settings.summary_path}")

## Create PHIA Baseline

In [ ]:
from pha.agents import create_phia_baseline

# Path to PHIA few-shot examples
# These are the original PHIA exemplars that teach the agent how to respond
FEW_SHOT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'few_shots', 'phia')
if not os.path.exists(FEW_SHOT_DIR):
    FEW_SHOT_DIR = os.path.join(os.getcwd(), 'few_shots', 'phia')

print(f"Few-shot directory: {FEW_SHOT_DIR}")
print(f"Few-shot notebooks available: {len([f for f in os.listdir(FEW_SHOT_DIR) if f.endswith('.ipynb')]) if os.path.exists(FEW_SHOT_DIR) else 0}")

In [ ]:
# Create the PHIA baseline
# NOTE: PHIA uses Gemini 2.5 Pro by default (matching original implementation)
# This is fixed for baseline comparison purposes

baseline = create_phia_baseline(
    gemini_api_key=GEMINI_API_KEY,
    tavily_api_key=TAVILY_API_KEY,
    settings=settings,
    few_shot_dir=FEW_SHOT_DIR,
    debug_verbose=True,
)

print(f"\n✓ PHIA baseline created")
print(f"  Model: {baseline._model_name}")
print(f"  Few-shots: {len([f for f in os.listdir(FEW_SHOT_DIR) if f.endswith('.ipynb')])} notebooks")

## Test Queries

Let's test the PHIA baseline with different types of queries.

### Query 1: Sleep Analysis (Data + Interpretation)

In [ ]:
query1 = "How has my sleep been over the past two weeks? Am I getting enough deep sleep?"

print(f"Query: {query1}")
print("="*60)

response1 = baseline.respond(query1)
print("\nResponse:")
print(response1)

### Query 2: Activity Analysis

In [ ]:
query2 = "What activities do I do most often and which ones burn the most calories?"

print(f"Query: {query2}")
print("="*60)

response2 = baseline.respond(query2)
print("\nResponse:")
print(response2)

### Query 3: Health Advice (Requires Web Search)

In [ ]:
query3 = "Based on my data, what can I do to improve my heart health?"

print(f"Query: {query3}")
print("="*60)

response3 = baseline.respond(query3)
print("\nResponse:")
print(response3)

## Comparison Notes

When comparing PHIA to PHA, consider:

| Metric | PHIA | PHA |
|--------|------|-------|
| Response Quality | Single perspective | Multi-agent synthesis |
| Data Analysis | General | Specialized DS agent |
| Medical Context | Basic | Domain expert agent |
| Actionable Advice | Generic | Health coach agent |
| Efficiency | Always full processing | Routes to relevant agents |

## Cleanup

In [ ]:
# Reset conversation state
baseline.reset_conversation()
print("Conversation state reset.")